In [ ]:
import os, subprocess
os.makedirs("build", exist_ok=True)

In [ ]:
# --- WaveDrom / VCD rendering utilities (auto-generated — do not edit) ---
import json, re, subprocess
from IPython.display import HTML, display

def _vcd_to_wavedrom(vcd_path, max_cycles=80, signals=None):
    """
    Minimal VCD → WaveDrom JSON converter.
    Works for single-bit and multi-bit signals from iverilog output.
    """
    with open(vcd_path) as f:
        vcd = f.read()

    # Parse variable declarations
    var_map = {}  # id_code → (name, width)
    for m in re.finditer(
        r"\$var\s+\w+\s+(\d+)\s+(\S+)\s+(\S+)", vcd
    ):
        width, code, name = int(m.group(1)), m.group(2), m.group(3)
        if signals is None or name in signals:
            var_map[code] = (name, width)

    if not var_map:
        print(f"⚠  No matching signals in {vcd_path}")
        return None

    # Parse value changes
    changes = {}  # id_code → [(time, value), ...]
    current_time = 0
    for line in vcd.splitlines():
        line = line.strip()
        if line.startswith("#"):
            current_time = int(line[1:])
        elif line.startswith("b"):
            parts = line.split()
            if len(parts) == 2 and parts[1] in var_map:
                val = int(parts[0][1:], 2) if parts[0][1:] else 0
                changes.setdefault(parts[1], []).append((current_time, val))
        elif len(line) >= 2 and line[0] in "01xXzZ" and line[1:] in var_map:
            code = line[1:]
            val = {"0": 0, "1": 1, "x": "x", "X": "x", "z": "z", "Z": "z"}[line[0]]
            changes.setdefault(code, []).append((current_time, val))

    # Determine time step (smallest non-zero delta)
    all_times = sorted({t for ch in changes.values() for t, _ in ch})
    if len(all_times) < 2:
        return None
    deltas = [all_times[i+1] - all_times[i] for i in range(len(all_times)-1) if all_times[i+1] != all_times[i]]
    if not deltas:
        return None
    step = min(deltas)
    end_time = min(all_times[-1], step * max_cycles) if max_cycles else all_times[-1]
    sample_times = list(range(0, end_time + 1, step))[:max_cycles]

    # Build WaveDrom signal list
    wave_signals = []
    for code, (name, width) in sorted(var_map.items(), key=lambda x: x[1][0]):
        ch = sorted(changes.get(code, []))
        # Sample at each time point
        samples = []
        cur_val = 0
        ci = 0
        for t in sample_times:
            while ci < len(ch) and ch[ci][0] <= t:
                cur_val = ch[ci][1]
                ci += 1
            samples.append(cur_val)

        if width == 1:
            # Single-bit: WaveDrom wave string
            wave_str = ""
            for v in samples:
                if v == "x":
                    wave_str += "x"
                elif v == "z":
                    wave_str += "z"
                else:
                    wave_str += str(int(v))
            wave_signals.append({"name": name, "wave": wave_str})
        else:
            # Multi-bit: use '=' for data with labels
            wave_str = ""
            data = []
            prev = None
            for v in samples:
                if v == prev:
                    wave_str += "."
                else:
                    wave_str += "="
                    data.append(f"0x{v:X}" if isinstance(v, int) else str(v))
                    prev = v
            wave_signals.append({"name": name, "wave": wave_str, "data": data})

    return {"signal": wave_signals, "config": {"hscale": 1}}


def show_waves(vcd_path="dump.vcd", max_cycles=80, signals=None, width=900):
    """Render VCD waveforms inline using WaveDrom."""
    wd = _vcd_to_wavedrom(vcd_path, max_cycles=max_cycles, signals=signals)
    if wd is None:
        print("No waveform data to display.")
        return
    wd_json = json.dumps(wd)
    html = f"""
    <div id="wd_{id(wd):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wd):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wd):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))


def show_wavedrom(wave_dict, width=900):
    """Render a raw WaveDrom JSON dict inline."""
    wd_json = json.dumps(wave_dict)
    html = f"""
    <div id="wd_{id(wave_dict):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wave_dict):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wave_dict):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))

print("✓ WaveDrom helpers loaded — use show_waves('dump.vcd') after simulation")

# Day 10 Lab: Numerical Architectures & Design Trade-offs

## Overview
Today you'll compare adder and multiplier architectures, implement a sequential
multiplier, work with fixed-point arithmetic, and produce your first structured
PPA comparison table. These are the same analysis habits you'll use in the
final project report.

## Prerequisites
- Completed Day 9 lab (memory)
- Pre-class videos on timing essentials, numerical architectures, and PPA watched
- Reuse your `hex_to_7seg.v` from Day 2 and `full_adder.v` from Day 2

## Exercises

| # | Exercise | Time | Key SLOs |
|---|----------|------|----------|
| 1 | Adder Architecture Comparison | 30 min | 10.1, 10.4 |
| 2 | Shift-and-Add Multiplier | 30 min | 10.2, 10.4 |
| 3 | Fixed-Point Arithmetic | 20 min | 10.3 |
| 4 | Timing Constraint Exercise | 10 min | 10.5 |
| 5 | PLL & CDC (Stretch) | 15 min | 10.5 |

## Deliverables
1. **Adder/multiplier PPA comparison table** with real data (LUTs, FFs, Fmax)
2. **Working shift-and-add multiplier** on FPGA with testbench
3. **Fixed-point demo** showing correct Q4.4 multiplication result on 7-seg

## PPA Comparison Table Template

| Module | Configuration | LUTs | FFs | Fmax (MHz) | Notes |
|--------|---------------|------|-----|------------|-------|
| Adder | Ripple-carry, 8-bit | | | | Manual chain |
| Adder | Ripple-carry, 16-bit | | | | Manual chain |
| Adder | Behavioral `+`, 8-bit | | | | Tool-chosen |
| Adder | Behavioral `+`, 16-bit | | | | Tool-chosen |
| Multiplier | Combinational `*`, 8-bit | | | | 1 cycle |
| Multiplier | Shift-and-add, 8-bit | | | | 8 cycles |

## Shared Resources
- `go_board.pcf` — Pin constraint file
- Reuse your `hex_to_7seg.v` and `full_adder.v` from previous labs

---
## Exercise Files

The starter files for each exercise are below. Edit the code, then run the simulation/build cells.

#### 📝 Exercise 1 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile adder_comparison.v
// =============================================================================
// adder_comparison.v — Adder Architecture Comparison
// Day 10, Exercise 1: Compare ripple-carry vs behavioral adder
// =============================================================================
// TASK: Implement both adder variants, synthesize at 8-bit and 16-bit,
//       and fill in the PPA comparison table in ppa_worksheet.md
//
// Commands:
//   yosys -p "read_verilog adder_comparison.v; synth_ice40 -top ripple_carry_8; stat"
//   yosys -p "read_verilog adder_comparison.v; synth_ice40 -top behavioral_8; stat"
//   (repeat for 16-bit variants)
//
//   For Fmax, add a registered wrapper and use nextpnr:
//   nextpnr-ice40 --hx1k --package vq100 --json adder.json --asc adder.asc --freq 25
// =============================================================================

// ---- Full adder (reuse from Day 2 or implement here) ----
module full_adder (
    input  wire i_a, i_b, i_cin,
    output wire o_sum, o_cout
);
    assign o_sum  = i_a ^ i_b ^ i_cin;
    assign o_cout = (i_a & i_b) | (i_b & i_cin) | (i_a & i_cin);
endmodule

// ---- Ripple-carry adder, 8-bit ----
module ripple_carry_8 (
    input  wire [7:0] i_a, i_b,
    input  wire       i_cin,
    output wire [7:0] o_sum,
    output wire       o_cout
);
    wire [7:0] carry;

    // TODO: Instantiate 8 full_adder modules, chaining carry signals
    // full_adder fa0 (.i_a(i_a[0]), .i_b(i_b[0]), .i_cin(i_cin),    .o_sum(o_sum[0]), .o_cout(carry[0]));
    // full_adder fa1 (.i_a(i_a[1]), .i_b(i_b[1]), .i_cin(carry[0]), .o_sum(o_sum[1]), .o_cout(carry[1]));
    // ... continue for all 8 bits

    // ---- YOUR CODE HERE ----

endmodule

// ---- Ripple-carry adder, 16-bit ----
module ripple_carry_16 (
    input  wire [15:0] i_a, i_b,
    input  wire        i_cin,
    output wire [15:0] o_sum,
    output wire        o_cout
);
    // TODO: Instantiate 16 full_adder modules (or chain two ripple_carry_8)

    // ---- YOUR CODE HERE ----

endmodule

// ---- Behavioral adder, 8-bit ----
module behavioral_8 (
    input  wire [7:0] i_a, i_b,
    input  wire       i_cin,
    output wire [8:0] o_sum
);
    assign o_sum = i_a + i_b + i_cin;
endmodule

// ---- Behavioral adder, 16-bit ----
module behavioral_16 (
    input  wire [15:0] i_a, i_b,
    input  wire        i_cin,
    output wire [16:0] o_sum
);
    assign o_sum = i_a + i_b + i_cin;
endmodule

In [ ]:
%%writefile Makefile
# Makefile — Adder Architecture Comparison
# Day 10, Exercise 1
#
# This exercise is primarily about synthesis comparison, not board programming.
# Use 'make stat_rc8' etc. to compare resource utilization across variants.

SRCS     = adder_comparison.v
PCF      = ../../go_board.pcf
DEVICE   = hx1k
PACKAGE  = vq100
IVERILOG_FLAGS = -g2012 -Wall

# ── Synthesis stats for each variant ──

stat_rc8:
	yosys -p "read_verilog $(SRCS); synth_ice40 -top ripple_carry_8; stat"

stat_rc16:
	yosys -p "read_verilog $(SRCS); synth_ice40 -top ripple_carry_16; stat"

stat_beh8:
	yosys -p "read_verilog $(SRCS); synth_ice40 -top behavioral_8; stat"

stat_beh16:
	yosys -p "read_verilog $(SRCS); synth_ice40 -top behavioral_16; stat"

stat_all: stat_rc8 stat_rc16 stat_beh8 stat_beh16

# ── Schematic comparison ──

show_rc8:
	yosys -p "read_verilog $(SRCS); synth_ice40 -top ripple_carry_8; show"

show_beh8:
	yosys -p "read_verilog $(SRCS); synth_ice40 -top behavioral_8; show"

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: stat_rc8 stat_rc16 stat_beh8 stat_beh16 stat_all show_rc8 show_beh8 clean

#### 📝 Exercise 2 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile shift_add_mult.v
// =============================================================================
// shift_add_mult.v — Sequential Shift-and-Add Multiplier
// Day 10, Exercise 2
// =============================================================================
// Implement an 8-bit unsigned multiplier using the shift-and-add algorithm.
// Architecture: FSM + shift register + accumulator
//
// Algorithm:
//   1. Load multiplicand and multiplier
//   2. For each bit of the multiplier (LSB first):
//      - If current multiplier bit is 1, add multiplicand to accumulator
//      - Shift multiplicand left (or shift multiplier right)
//   3. After 8 iterations, accumulator holds the product
//
// Compare resource usage to: assign product = a * b; (combinational)
// =============================================================================

module shift_add_mult (
    input  wire        i_clk,
    input  wire        i_rst,
    input  wire        i_start,    // pulse to begin multiplication
    input  wire [7:0]  i_a,        // multiplicand
    input  wire [7:0]  i_b,        // multiplier
    output reg  [15:0] o_product,  // result
    output reg         o_done,     // pulse when complete
    output reg         o_busy      // high during computation
);

    // FSM states
    localparam IDLE    = 2'b00;
    localparam COMPUTE = 2'b01;
    localparam DONE    = 2'b10;

    reg [1:0]  r_state;
    reg [15:0] r_accumulator;
    reg [7:0]  r_multiplicand_shifted; // or use a wider register
    reg [7:0]  r_multiplier;
    reg [3:0]  r_bit_count;  // counts 0 to 7

    // TODO: Implement the FSM and datapath
    //
    // IDLE state:
    //   - Wait for i_start
    //   - On i_start: load operands, clear accumulator, go to COMPUTE
    //
    // COMPUTE state:
    //   - Check LSB of r_multiplier
    //   - If 1: add r_multiplicand_shifted to r_accumulator
    //   - Shift r_multiplier right by 1
    //   - Shift r_multiplicand_shifted left by 1 (or use a wider shift)
    //   - Increment r_bit_count
    //   - After 8 iterations: go to DONE
    //
    // DONE state:
    //   - Output the product
    //   - Pulse o_done
    //   - Return to IDLE
    //
    // Hint: You can use a 16-bit register for the shifted multiplicand
    //       to avoid overflow: reg [15:0] r_mcand;

    // ---- YOUR CODE HERE ----

endmodule

// ---- For comparison: combinational multiplier ----
module comb_mult (
    input  wire [7:0]  i_a, i_b,
    output wire [15:0] o_product
);
    assign o_product = i_a * i_b;
endmodule

In [ ]:
%%writefile tb_shift_add_mult.v
// =============================================================================
// tb_shift_add_mult.v — Testbench for Shift-and-Add Multiplier
// Day 10, Exercise 2
// =============================================================================
`timescale 1ns/1ps

module tb_shift_add_mult;

    reg        clk, rst, start;
    reg  [7:0] a, b;
    wire [15:0] product;
    wire        done, busy;

    shift_add_mult uut (
        .i_clk(clk), .i_rst(rst), .i_start(start),
        .i_a(a), .i_b(b),
        .o_product(product), .o_done(done), .o_busy(busy)
    );

    // Clock: 10 ns period
    always #5 clk = ~clk;

    integer pass_count, fail_count;
    reg [15:0] expected;

    task test_multiply;
        input [7:0] ta, tb;
        begin
            a = ta;
            b = tb;
            expected = ta * tb;
            @(posedge clk);
            start = 1;
            @(posedge clk);
            start = 0;

            // Wait for done
            wait (done == 1);
            @(posedge clk);

            if (product === expected) begin
                $display("PASS: %0d * %0d = %0d", ta, tb, product);
                pass_count = pass_count + 1;
            end else begin
                $display("FAIL: %0d * %0d = %0d (expected %0d)", ta, tb, product, expected);
                fail_count = fail_count + 1;
            end

            // Wait a cycle before next test
            @(posedge clk);
        end
    endtask

    initial begin
        $dumpfile("shift_add_mult.vcd");
        $dumpvars(0, tb_shift_add_mult);

        clk = 0; rst = 1; start = 0; a = 0; b = 0;
        pass_count = 0; fail_count = 0;

        // Reset
        repeat (3) @(posedge clk);
        rst = 0;
        @(posedge clk);

        // Test cases
        test_multiply(8'd0,   8'd0);     // 0 * 0
        test_multiply(8'd1,   8'd1);     // 1 * 1
        test_multiply(8'd0,   8'd100);   // 0 * N
        test_multiply(8'd100, 8'd0);     // N * 0
        test_multiply(8'd1,   8'd255);   // 1 * max
        test_multiply(8'd255, 8'd1);     // max * 1
        test_multiply(8'd3,   8'd7);     // 3 * 7 = 21
        test_multiply(8'd12,  8'd13);    // 12 * 13 = 156
        test_multiply(8'd15,  8'd15);    // 15 * 15 = 225
        test_multiply(8'd255, 8'd255);   // max * max = 65025

        // Summary
        $display("");
        $display("========================================");
        $display("  %0d/%0d tests passed", pass_count, pass_count + fail_count);
        if (fail_count == 0)
            $display("  ALL TESTS PASSED");
        else
            $display("  %0d TESTS FAILED", fail_count);
        $display("========================================");

        $finish;
    end

endmodule

In [ ]:
%%writefile Makefile
# Makefile — Shift-and-Add Multiplier
# Day 10, Exercise 2

PROJECT  = shift_add_mult
TOP      = shift_add_mult
PCF      = ../../go_board.pcf
SRCS     = shift_add_mult.v
TB       = tb_shift_add_mult.v

DEVICE   = hx1k
PACKAGE  = vq100
IVERILOG_FLAGS = -g2012 -Wall

all: sim

sim: $(TB) $(SRCS)
	iverilog $(IVERILOG_FLAGS) -o sim.vvp $(TB) $(SRCS)
	vvp sim.vvp
	@echo "=== Simulation complete ==="

wave: sim
	gtkwave shift_add_mult.vcd &

stat_seq:
	yosys -p "read_verilog $(SRCS); synth_ice40 -top shift_add_mult; stat"

stat_comb:
	yosys -p "read_verilog $(SRCS); synth_ice40 -top comb_mult; stat"

stat_all: stat_seq stat_comb

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: all sim wave stat_seq stat_comb stat_all clean

In [ ]:
# Compile and simulate
!iverilog -g2012 -Wall -o sim.vvp tb_shift_add_mult.v shift_add_mult.v && vvp sim.vvp

In [ ]:
show_waves('dump.vcd')

#### 📝 Exercise 3 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile fixed_point.v
// =============================================================================
// fixed_point.v — Fixed-Point Q4.4 Arithmetic
// Day 10, Exercise 3
// =============================================================================
// Q4.4 format: 4 integer bits, 4 fractional bits = 8 bits total
// Value = raw_integer / 16
// Example: 8'b0010_1000 = 40 decimal → 40/16 = 2.5
//
// TASK:
//   1. Implement Q4.4 addition (straightforward — just add)
//   2. Implement Q4.4 multiplication (Q4.4 × Q4.4 = Q8.8, then extract Q4.4)
//   3. Display the integer part of the result on 7-seg
//   4. Test with known values: 2.5 × 3.0 = 7.5 → integer part = 7
// =============================================================================

module fixed_point_demo (
    input  wire        i_clk,
    input  wire [7:0]  i_a,      // Q4.4 input A
    input  wire [7:0]  i_b,      // Q4.4 input B
    output wire [7:0]  o_sum,    // Q4.4 sum
    output wire [7:0]  o_prod,   // Q4.4 product (truncated from Q8.8)
    output wire [3:0]  o_prod_int // integer part of product (for 7-seg)
);

    // ---- Addition ----
    // Q4.4 + Q4.4 = Q4.4 (with potential overflow into bit 8)
    // For simplicity, we'll ignore overflow in this exercise
    // TODO: implement the addition
    // assign o_sum = ???;

    // ---- YOUR CODE HERE ----


    // ---- Multiplication ----
    // Q4.4 × Q4.4 = Q8.8 (16 bits: 8 integer, 8 fractional)
    // To get Q4.4 result: extract bits [11:4] from the 16-bit product
    //
    //   Full product bits: [15:8] = integer part (Q8)
    //                      [7:0]  = fractional part (.8)
    //   But we want Q4.4:  [11:8] = 4 integer bits
    //                      [7:4]  = 4 fractional bits
    //   So: o_prod = full_product[11:4]

    wire [15:0] w_full_product;

    // TODO: compute the full product and extract Q4.4 result
    // assign w_full_product = ???;
    // assign o_prod = ???;

    // ---- YOUR CODE HERE ----


    // ---- Integer part extraction ----
    // The integer part of a Q4.4 number is the upper 4 bits
    // TODO: extract integer part of the product for 7-seg display
    // assign o_prod_int = ???;

    // ---- YOUR CODE HERE ----

endmodule

In [ ]:
%%writefile tb_fixed_point.v
// =============================================================================
// tb_fixed_point.v — Testbench for Q4.4 Fixed-Point Arithmetic
// Day 10, Exercise 3
// =============================================================================
`timescale 1ns/1ps

module tb_fixed_point;

    reg        clk;
    reg  [7:0] a, b;
    wire [7:0] sum, prod;
    wire [3:0] prod_int;

    fixed_point_demo uut (
        .i_clk(clk), .i_a(a), .i_b(b),
        .o_sum(sum), .o_prod(prod), .o_prod_int(prod_int)
    );

    always #5 clk = ~clk;

    integer pass_count, fail_count;

    // Helper: convert Q4.4 to real for display
    // Q4.4 value = raw / 16.0
    task test_fixed;
        input [7:0] ta, tb;
        input [3:0] expected_int;  // expected integer part of product
        real ra, rb, rprod;
        begin
            a = ta;
            b = tb;
            #20;  // let combinational logic settle

            ra = ta / 16.0;
            rb = tb / 16.0;
            rprod = ra * rb;

            if (prod_int === expected_int) begin
                $display("PASS: %.2f * %.2f = %.2f (int part = %0d, got %0d)",
                         ra, rb, rprod, expected_int, prod_int);
                pass_count = pass_count + 1;
            end else begin
                $display("FAIL: %.2f * %.2f = %.2f (int part expected %0d, got %0d, raw prod=0x%02h)",
                         ra, rb, rprod, expected_int, prod_int, prod);
                fail_count = fail_count + 1;
            end
        end
    endtask

    initial begin
        $dumpfile("fixed_point.vcd");
        $dumpvars(0, tb_fixed_point);

        clk = 0;
        pass_count = 0;
        fail_count = 0;

        // Q4.4 encoding: value * 16
        // 2.5  = 40  = 8'h28
        // 3.0  = 48  = 8'h30
        // 1.0  = 16  = 8'h10
        // 0.5  = 8   = 8'h08
        // 4.25 = 68  = 8'h44
        // 7.5  → int part = 7

        test_fixed(8'h28, 8'h30, 4'd7);   // 2.5 * 3.0 = 7.5  → int 7
        test_fixed(8'h10, 8'h10, 4'd1);   // 1.0 * 1.0 = 1.0  → int 1
        test_fixed(8'h20, 8'h20, 4'd4);   // 2.0 * 2.0 = 4.0  → int 4
        test_fixed(8'h30, 8'h10, 4'd3);   // 3.0 * 1.0 = 3.0  → int 3
        test_fixed(8'h08, 8'h08, 4'd0);   // 0.5 * 0.5 = 0.25 → int 0
        test_fixed(8'h44, 8'h20, 4'd8);   // 4.25 * 2.0 = 8.5 → int 8
        test_fixed(8'h00, 8'hFF, 4'd0);   // 0.0 * max  = 0   → int 0

        $display("");
        $display("========================================");
        $display("  %0d/%0d tests passed", pass_count, pass_count + fail_count);
        if (fail_count == 0) $display("  ALL TESTS PASSED");
        else $display("  %0d TESTS FAILED", fail_count);
        $display("========================================");

        $finish;
    end

endmodule

In [ ]:
%%writefile Makefile
# Makefile — Fixed-Point Arithmetic
# Day 10, Exercise 3

SRCS     = fixed_point.v
TB       = tb_fixed_point.v
IVERILOG_FLAGS = -g2012 -Wall

sim: $(TB) $(SRCS)
	iverilog $(IVERILOG_FLAGS) -o sim.vvp $(TB) $(SRCS)
	vvp sim.vvp
	@echo "=== Simulation complete ==="

wave: sim
	gtkwave fixed_point.vcd &

stat:
	yosys -p "read_verilog $(SRCS); synth_ice40 -top fixed_point_demo; stat"

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: sim wave stat clean

In [ ]:
# Compile and simulate
!iverilog -g2012 -Wall -o sim.vvp tb_fixed_point.v fixed_point.v && vvp sim.vvp

In [ ]:
show_waves('dump.vcd')

#### 📝 Exercise 5 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile top_cdc_demo.v
// =============================================================================
// top_cdc_demo.v — Clock Domain Crossing Demo (Stretch Exercise)
// Day 10, Exercise 5
// =============================================================================
// Debounce a button in the 25 MHz domain, synchronize into the 50 MHz PLL
// domain, count button presses, and display on 7-seg.

module top_cdc_demo (
    input  wire i_clk,       // 25 MHz
    input  wire i_switch1,   // button
    output wire o_segment1_a, o_segment1_b, o_segment1_c,
    output wire o_segment1_d, o_segment1_e, o_segment1_f, o_segment1_g,
    output wire o_led1, o_led2, o_led3, o_led4
);

    // PLL: 50 MHz
    wire w_pll_clk, w_pll_locked;
    SB_PLL40_CORE #(
        .FEEDBACK_PATH("SIMPLE"),
        .DIVR(4'b0000), .DIVF(7'b0011111),
        .DIVQ(3'b100),  .FILTER_RANGE(3'b010)
    ) pll_inst (
        .REFERENCECLK(i_clk), .PLLOUTCORE(w_pll_clk),
        .LOCK(w_pll_locked), .RESETB(1'b1), .BYPASS(1'b0)
    );

    // 25 MHz: debounce the button (reuse your debounce module)
    wire w_btn_clean;
    debounce #(.CLKS_TO_STABLE(250_000)) db (
        .i_clk(i_clk), .i_bouncy(i_switch1), .o_clean(w_btn_clean)
    );
    wire w_btn_active = ~w_btn_clean;

    // TODO: 2-FF synchronizer: 25 MHz → 50 MHz
    // TODO: Edge detector in PLL domain
    // TODO: 4-bit counter in PLL domain (reset when !locked)
    // TODO: Synchronize count back to 25 MHz for display
    // TODO: Instantiate hex_to_7seg for display

    // ---- YOUR CODE HERE ----

    assign o_led1 = ~w_btn_active;
    assign o_led2 = ~w_pll_locked;
    assign o_led3 = 1'b1;
    assign o_led4 = 1'b1;

endmodule

In [ ]:
%%writefile top_pll_demo.v
// =============================================================================
// top_pll_demo.v — PLL Clock Generation Demo (Stretch Exercise)
// Day 10, Exercise 5
// =============================================================================
// Use icepll to find parameters: icepll -i 25 -o 50

module top_pll_demo (
    input  wire i_clk,       // 25 MHz
    output wire o_led1,      // blinks from 25 MHz domain
    output wire o_led2,      // blinks from PLL domain
    output wire o_led3,      // PLL lock indicator
    output wire o_led4       // heartbeat (25 MHz domain)
);

    wire w_pll_clk, w_pll_locked;

    SB_PLL40_CORE #(
        .FEEDBACK_PATH("SIMPLE"),
        // TODO: Run `icepll -i 25 -o 50` and fill in these parameters
        .DIVR(4'b0000),
        .DIVF(7'b0000000),     // TODO
        .DIVQ(3'b000),         // TODO
        .FILTER_RANGE(3'b000)  // TODO
    ) pll_inst (
        .REFERENCECLK(i_clk),
        .PLLOUTCORE(w_pll_clk),
        .LOCK(w_pll_locked),
        .RESETB(1'b1),
        .BYPASS(1'b0)
    );

    // 25 MHz domain blinker
    reg [23:0] r_count_25;
    always @(posedge i_clk)
        r_count_25 <= r_count_25 + 1;
    assign o_led1 = ~r_count_25[23];

    // TODO: PLL domain blinker
    //   - Reset when PLL is not locked
    //   - Use bit [24] for ~1.5 Hz at 50 MHz
    //   - Assign to o_led2 (active-low)

    // ---- YOUR CODE HERE ----

    assign o_led3 = ~w_pll_locked;
    assign o_led4 = ~r_count_25[22];

endmodule

In [ ]:
%%writefile Makefile
# Makefile — PLL & CDC Stretch Exercises
# Day 10, Exercise 5

PCF      = ../../go_board.pcf
DEVICE   = hx1k
PACKAGE  = vq100

pll: top_pll_demo.v
	yosys -p "synth_ice40 -top top_pll_demo -json pll.json" $<
	nextpnr-ice40 --$(DEVICE) --package $(PACKAGE) --pcf $(PCF) --json pll.json --asc pll.asc --freq 50
	icepack pll.asc pll.bin

prog_pll: pll
	iceprog pll.bin

cdc: top_cdc_demo.v hex_to_7seg.v debounce.v
	yosys -p "synth_ice40 -top top_cdc_demo -json cdc.json" $^
	nextpnr-ice40 --$(DEVICE) --package $(PACKAGE) --pcf $(PCF) --json cdc.json --asc cdc.asc --freq 50
	icepack cdc.asc cdc.bin

prog_cdc: cdc
	iceprog cdc.bin

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: pll prog_pll cdc prog_cdc clean